Develop a single Real Time example that complies to all Solid Principles
Also have the same example with violations of SOLID.

S → Single responsibility
    "One class → one main job"

O → Open/Closed
    "Add new functionality without breaking/modifying existing functionality"

L → Liskov Substitution
    "Child should properly replace parent"

I → Interface Segregation
    "Don't force unnecessary methods"

D → Dependency Inversion
    "Depend on abstraction, not concrete class"

Real-Time Example: Online Food Ordering System

In [1]:
// Version 1 — SOLID Principles Violated
class OrderService {

    void placeOrder(String customer, double amount, String paymentType) {

        // Calculate bill
        double tax = amount * 0.05;
        double total = amount + tax;

        System.out.println("Customer: " + customer);
        System.out.println("Total Bill: " + total);

        // Payment
        if (paymentType.equals("UPI")) {
            System.out.println("Payment through UPI");
        } else if (paymentType.equals("CARD")) {
            System.out.println("Payment through Card");
        }

        // Save order
        System.out.println("Saving order to MySQL database");

        // Send notification
        System.out.println("Sending Email notification");

        // Generate invoice
        System.out.println("Generating invoice");
    }
}

1. SRP ❌ OrderService is doing many jobs:
2. OCP ❌ Suppose we add Credit Card, PayPal, Cash, Net Banking
We have to keep modifying: if (paymentType.equals(...))
3. LSP ❌ The design doesn't properly use substitutable abstractions for different payment/notification implementations.
There is no common abstraction such as: Payment
4. ISP ❌ There are no small, focused interfaces. Everything is concentrated inside one large service.
5. DIP ❌ This code directly depends on concrete technologies:
MySQL
Email
UPI


Version 2 — SOLID-Compliant

In [ ]:
// 1. Order — Data responsibility
class Order {

    private String customerName;
    private double amount;

    Order(String customerName, double amount) {
        this.customerName = customerName;
        this.amount = amount;
    }

    public String getCustomerName() {
        return customerName;
    }

    public double getAmount() {
        return amount;
    }
}


In [ ]:
// 2. SRP — Separate responsibilities
class BillCalculator {

    double calculateTotal(Order order) {
        double tax = order.getAmount() * 0.05;
        return order.getAmount() + tax;
    }
}
// Invoice
class InvoiceGenerator {

    void generate(Order order, double total) {
        System.out.println("Invoice generated for " + order.getCustomerName());
        System.out.println("Total: " + total);
    }
}
// Database
interface OrderRepository {
    void save(Order order);
}

// Implementation:

class MySQLOrderRepository implements OrderRepository {

    public void save(Order order) {
        System.out.println("Order saved to MySQL");
    }
}

In [ ]:
// 3. OCP — Payment abstraction
interface Payment {

    void pay(double amount);
}

// UPI:

class UPIPayment implements Payment {

    public void pay(double amount) {
        System.out.println("Paid ₹" + amount + " using UPI");
    }
}

// Card:

class CardPayment implements Payment {

    public void pay(double amount) {
        System.out.println("Paid ₹" + amount + " using Card");
    }
}

// Now we can add:

class CashPayment implements Payment {

    public void pay(double amount) {
        System.out.println("Paid ₹" + amount + " using Cash");
    }
}

In [ ]:
// 4. ISP — Separate notification interfaces
interface EmailNotification {
    void sendEmail(String message);
}
interface SMSNotification {
    void sendSMS(String message);
}

In [ ]:
// 5. LSP — Payment implementations
Payment payment = new UPIPayment();
payment.pay(1050);

payment = new CardPayment();
payment.pay(1050);

payment = new CashPayment();
payment.pay(1050);

In [ ]:
// 6. DIP — OrderService depends on abstractions
class OrderService {

    private BillCalculator billCalculator;
    private Payment payment;
    private OrderRepository repository;
    private EmailNotification notification;
    private InvoiceGenerator invoiceGenerator;

    OrderService(BillCalculator billCalculator,
                 Payment payment,
                 OrderRepository repository,
                 EmailNotification notification,
                 InvoiceGenerator invoiceGenerator) {

        this.billCalculator = billCalculator;
        this.payment = payment;
        this.repository = repository;
        this.notification = notification;
        this.invoiceGenerator = invoiceGenerator;
    }

    void placeOrder(Order order) {

        double total = billCalculator.calculateTotal(order);

        payment.pay(total);

        repository.save(order);

        invoiceGenerator.generate(order, total);

        notification.sendEmail(
                "Order placed successfully for " + order.getCustomerName()
        );
    }
}